<a href="https://colab.research.google.com/github/j-c-stuifbergen/red_blood_cell_mesh/blob/main/Abaqus_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
useGoogleDrive = True

if useGoogleDrive:
  #@title important: set the path to the data files
  from google.colab import drive
  drive.mount('/content/drive')
  sourcePath = '/content/drive/MyDrive/BM41090 mechanical simulation group assignment/'
else:
  sourcePath = './'

fileName ="rbc_biconcave_half.inp"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [6]:
"""
RBC Biconcave Mesh Generator - Student Edition (<= 1000 nodes)
N_phi=30, N_r=16 → 962 nodes, 990 elements  (best quality within limit)
"""
import numpy as np

R0 = 3.91;  C0 = 0.207161;  C1 = 2.002558;  C2 = -1.122762

def Z_rbc(r):
    if r >= R0 or r < 0: return 0.0
    rr = (r/R0)**2
    return 0.5*R0*np.sqrt(max(0.0,1.0-rr))*(C0+C1*rr+C2*rr**2)

phi_max =  1.0 * np.pi # max 2 * np.pi
N_phi = 20 # circumferential angles, including endpoint
N_r   = 16 # radial positions, excluding the center →  962 nodes, 990 elements
N_h   = 5 # on the flat edges, between upper and lower rings
nodes = []
node_id = 1

# Apex nodes
z_apex = Z_rbc(0.0)
nodes.append((0.0, 0.0, -z_apex)); apex_bot = node_id; node_id+=1
nodes.append((0.0, 0.0,  z_apex)); apex_top = node_id; node_id+=1

# add the first nodes of the side (these nodes are shared by two sides)
rh=[apex_bot];
h_values = np.linspace(-z_apex, z_apex, N_h+1, endpoint=True)
for h in h_values[1:-1]: # skip apex_bot and apex_top
        nodes.append((0,0,h)); rh.append(node_id); node_id+=1
rh.append(apex_top)
right_side=[rh] ; left_side=[rh]

# Rim nodes
r_values   = np.linspace(R0/N_r, R0, N_r-2, endpoint=False)
# use a smaller interval at the end
r_values = np.append(r_values,  [0.3*r_values[-1]+0.7*R0, R0])

phi_values = np.linspace(0, phi_max, N_phi, endpoint=True)

top_rings = {}; bot_rings = {}
for i_r, r in enumerate(r_values):
    z = Z_rbc(r)
    if N_r-1 == i_r:
      z=0 # prevent round-off when r = R0
    h_values = np.linspace(-z, z, N_h+1, endpoint=True)
    tr=[]; br=[]
    nodeCount_right = node_id # store node of 0=phi, -z
    for phi in phi_values:
        x=r*np.cos(phi); y=r*np.sin(phi)
        nodes.append((x,y,-z)); br.append(node_id)
        nodeCount_left = node_id # store node of phi_max, -z
        if i_r < N_r-1: # different node for +z
          node_id+=1;  nodes.append((x,y, z));
        # else: z = 0, use the same node
        tr.append(node_id); node_id+=1
    top_rings[i_r]=tr; bot_rings[i_r]=br

    # append the node of minimal z of this r
    rh=[nodeCount_right]; # node of phi = 0
    lh=[nodeCount_left] # = node of phi_max
    # max and min nodes have already been defined
    # Define the intermediate nodes
    if i_r < N_r-1:
      for h in h_values[1:-1]:
          # for 0 = phi :
          nodes.append((r,0, h)); rh.append(node_id); node_id+=1
          # for phi = phi_max
          nodes.append((x,y, h)); lh.append(node_id); node_id+=1
      rh.append(nodeCount_right+1);
      lh.append(nodeCount_left+1)
      #left_side[i_r] = lh; right_side[i_r] = rh
      left_side.append(lh); right_side.append(rh)
    else: # z=0
      apex_right = nodeCount_right
      apex_left  = nodeCount_left

# Elements
ringElts=[]
sideElts=[]
def quad(a,b,c,d): return (a,b,c,d)

for i_r in range(N_r):
    ro_t=top_rings[i_r]; ro_b=bot_rings[i_r]
    if i_r==0:
        for p in range(N_phi-1):
            ringElts.append((apex_top, ro_t[p], ro_t[(p+1)%N_phi], ro_t[(p+1)%N_phi]))
            ringElts.append((apex_bot, ro_b[(p+1)%N_phi], ro_b[p], ro_b[p]))
    else:
        ri_t=top_rings[i_r-1]; ri_b=bot_rings[i_r-1]
        for p in range(N_phi-1):
            ringElts.append(quad(ri_t[p], ri_t[(p+1)%N_phi], ro_t[(p+1)%N_phi], ro_t[p]))
            ringElts.append(quad(ri_b[p], ro_b[p], ro_b[(p+1)%N_phi], ri_b[(p+1)%N_phi]))

""" This section makes connections between upper and lower rings,
    but both are already at z=0. So this is nonsense!
tr=top_rings[N_r-1]; br=bot_rings[N_r-1]
 for p in range(N_phi):
    ringElts.append(quad(tr[p], tr[(p+1)%N_phi], br[(p+1)%N_phi], br[p]))
"""
# create elements for the sides
for i_r in range(N_r):
  if N_r-1==i_r:
      # make triangular grids, because z=0
      rih = right_side[i_r]; lih = left_side[i_r]
      for h in range(N_h):
        # hoping that I choose the correct orientation,
        # and that the order of the elements is not important
        sideElts.append(quad(lih[h],apex_left,apex_left,lih[h+1]))
        sideElts.append(quad(apex_right,rih[h],rih[h+1],apex_right))
  else:
      # rh.shape[0] = lh.shape[0] = N_r + 1
      # right-inner-height, right-outer-height etc.
      rih = right_side[i_r]; lih = left_side[i_r]
      roh = right_side[i_r+1]; loh = left_side[i_r+1]
      for h in range(N_h):
        # hoping that I choose the correct orientation,
        # and that the order of the elements is not important
        sideElts.append(quad(lih[h],loh[h],loh[h+1],lih[h+1]))
        sideElts.append(quad(roh[h],rih[h],rih[h+1],roh[h+1]))

print(f"Nodes:    {len(nodes)}  (limit: 1000)")
print(f"Elements: {len(ringElts)+len(sideElts)}")

# Write .inp
with open(sourcePath+fileName,"w") as f:
    f.write("** RBC Biconcave Shell pizza slice - Student Edition (962 nodes / 990 elements)\n")
    f.write("** Shape: Evans & Fung (1972) as in Dao et al. (2003) Eq. 7\n")
    f.write("** Units: micrometers (µm), picoNewtons (pN)\n")
    f.write("** SI equivalent: 1 pN/µm² = 1 µPa; C10 in Pa = C10*1e-12 pN/µm²\n")
    f.write("** NOTE: material values below are in SI (Pa) — use SI units throughout\n")
    f.write("**\n*Heading\nRBC biconcave shell model - student edition\n**\n")

    f.write("*Node\n")
    for i,(x,y,z) in enumerate(nodes):
        f.write(f"{i+1}, {x:.7f}, {y:.7f}, {z:.7f}\n")

    f.write("**\n*Element, type=S4R\n")
    i_element = 0
    for i,e in enumerate(ringElts):
    # for e in ringElts:
        i_element += 1
        f.write(f"{i_element}, {e[0]}, {e[1]}, {e[2]}, {e[3]}\n")
    f.write("**\n** ════════════════════════════════════════════════\n")
    f.write("** side ELEMENTS\n")
    f.write(f"** {i_element}= i_element════════════════════════════════════════════════\n")
    for j,e in enumerate(sideElts):
        i_element += 1
        f.write(f"{i_element}, {e[0]}, {e[1]}, {e[2]}, {e[3]}\n")

    f.write("**\n*Elset, elset=RBC_MEMBRANE, generate\n")
    f.write(f"1, {len(ringElts)}, 1\n")
    f.write("**\n*Elset, elset=RBC_SIDES, generate\n")
    f.write(f"{len(ringElts)+1}, {len(ringElts)+len(sideElts)}, 1\n")
    f.write("**\n*Nset, nset=ALL_NODES, generate\n")
    f.write(f"1, {len(nodes)}, 1\n")

    # Symmetry node sets (nodes near Z=0 plane, i.e. rim nodes)
    f.write("**\n** Rim nodes (r=R0, Z~0) - useful for BCs\n")
    rim_nodes = top_rings[N_r-1] # = bot_rings[N_r-1]
    f.write("*Nset, nset=RIM_NODES\n")
    line=[];
    for n in rim_nodes:
        line.append(str(n))
        if len(line)==16: f.write(", ".join(line)+"\n"); line=[]
    if line: f.write(", ".join(line)+"\n")

    # Shell sections
    f.write("**\n** ════════════════════════════════════════════════\n")
    f.write("** HEALTHY RBC SECTION\n")
    f.write("** ════════════════════════════════════════════════\n")
    f.write("*Shell Section, elset=RBC_MEMBRANE, material=RBC_HEALTHY\n")
    f.write("** thickness (m), integration points\n")
    f.write("82e-9, 5\n")
    f.write("** ════════════════════════════════════════════════\n")
    f.write("*Shell Section, elset=RBC_SIDES, material=RBC_INTERIOR\n")
    f.write("** thickness (m), integration points\n")
    f.write("82e-9, 5\n")

    f.write("**\n** ════════════════════════════════════════════════\n")
    f.write("** MATERIALS\n")
    f.write("** ════════════════════════════════════════════════\n")

    f.write("**\n*Material, name=RBC_INTERIOR\n")
    f.write("** Neo-Hookean: C10=137 Pa, D1=7.3e-5 Pa^-1\n")
    f.write("** properties still to be defined")
    f.write("*Hyperelastic, neo hooke\n")
    f.write("137.0, 7.3e-5\n")

    f.write("**\n*Material, name=RBC_HEALTHY\n")
    f.write("** Neo-Hookean: C10=137 Pa, D1=7.3e-5 Pa^-1\n")
    f.write("** Source: Dao et al. (2003), mu0=22.5 uN/m, h0=82nm\n")
    f.write("*Hyperelastic, neo hooke\n")
    f.write("137.0, 7.3e-5\n")

    f.write("**\n*Material, name=RBC_INFECTED\n")
    f.write("** Neo-Hookean: C10=750 Pa, D1=1.3e-5 Pa^-1\n")
    f.write("** Source: Lim (2006), mu0=150 uN/m, h0=100nm\n")
    f.write("*Hyperelastic, neo hooke\n")
    f.write("750.0, 1.3e-5\n")

    f.write("**\n** ════════════════════════════════════════════════\n")
    f.write("** TO SWITCH TO INFECTED: change shell section above to:\n")
    f.write("** *Shell Section, elset=RBC_MEMBRANE, material=RBC_INFECTED\n")
    f.write("** and change thickness to 100e-9\n")
    f.write("** ════════════════════════════════════════════════\n")

print(f"\nFile written: "+ fileName)
print(f"\nShape verification:")
print(f"  Z(r=0)     = {Z_rbc(0):.4f} µm  (dimple depth)")
print(f"  Z(r=R0/2)  = {Z_rbc(R0/2):.4f} µm")
print(f"  Z(r=R0)    = {Z_rbc(R0):.4f} µm  (rim, should be 0)")
print(f"  Cell diameter = {2*R0:.2f} µm")



Nodes:    746  (limit: 1000)
Elements: 768

File written: rbc_biconcave_half.inp

Shape verification:
  Z(r=0)     = 0.4050 µm  (dimple depth)
  Z(r=R0/2)  = 1.0796 µm
  Z(r=R0)    = 0.0000 µm  (rim, should be 0)
  Cell diameter = 7.82 µm
